<a href="https://colab.research.google.com/github/EhsanGhasemi423/INFS-8368/blob/main/5_AutoGrad_Automatic_Differenciation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Automatic Differentiation

In the previous section, we learned that training a neural network requires computing many derivatives.

For a small example, we can compute derivatives by hand.

However, modern neural networks may contain **thousands or even millions of trainable parameters**, making manual differentiation impossible.

Fortunately, TensorFlow performs these calculations automatically using **automatic differentiation** (often called **autograd**).

---

### How Does It Work?

During the **forward pass**, TensorFlow records every mathematical operation performed by the model.

This creates a **computational graph** that describes how the output depends on the model parameters.

During the **backward pass**, TensorFlow works backward through this graph, repeatedly applying the **chain rule** to compute the gradient of the loss with respect to every trainable parameter.

This process is called **backpropagation**.

---

### Training Workflow

1. Perform a forward pass to make predictions.
2. Compute the loss.
3. TensorFlow automatically computes the gradients using automatic differentiation.
4. The optimizer updates the model parameters.
5. Repeat until the loss is minimized.

In [ ]:
import tensorflow as tf

**Example**

Let's see how TensorFlow computes derivatives automatically using a simple function.

Suppose we define the following function:

$$
y = 2x^2.
$$

Our goal is to compute the derivative of \(y\) with respect to \(x\).

Instead of calculating the derivative by hand, we will let **TensorFlow** compute it automatically using **automatic differentiation**.

First, we assign an initial value to \(x\).

In [ ]:
x = tf.range(4, dtype=tf.float32)
x

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([0., 1., 2., 3.], dtype=float32)>

Before TensorFlow can compute gradients, it must know which variables should be tracked.

We therefore create a TensorFlow variable:

```python
x = tf.Variable(x)
```

Unlike a regular tensor, a `tf.Variable` is **trainable**, meaning TensorFlow can automatically compute gradients with respect to it.

In deep learning, the model's **weights** and **biases** are stored as `tf.Variable` objects because they are updated during training.

In [ ]:
x = tf.Variable(x)

Next, we define our function and compute its value.

At the same time, TensorFlow records every mathematical operation performed during this computation.

This recording is called a **GradientTape**. It creates a computational graph that keeps track of how the output depends on the input.

Later, TensorFlow will use this recorded information to automatically compute the gradient using the chain rule.

In [ ]:
# Record all computations onto a tape
with tf.GradientTape() as t:
    y = 2 * tf.tensordot(x, x, axes=1)
y

<tf.Tensor: shape=(), dtype=float32, numpy=28.0>

We can now calculate the gradient of y with respect to x by calling the gradient method.

In [ ]:
x_grad = t.gradient(y, x)
x_grad

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([ 0.,  4.,  8., 12.], dtype=float32)>

Next, we compare the gradient computed by TensorFlow with the expected result.

For the function

$$
y = 2\mathbf{x}^{\top}\mathbf{x},
$$

the expected gradient is

$$
\nabla_{\mathbf{x}} y = 4\mathbf{x}.
$$

If TensorFlow is working correctly, the gradient computed by `GradientTape` should match this expected result.

In [ ]:
x_grad == 4 * x

<tf.Tensor: shape=(4,), dtype=bool, numpy=array([ True,  True,  True,  True])>

Next, we compute the gradient of a different function.

TensorFlow automatically computes the gradient for the new function using a new `GradientTape`.

The new gradient depends on the function being differentiated, so it is generally different from the gradient computed for the previous function.

In [ ]:
with tf.GradientTape() as t:
    y = tf.reduce_sum(x)
t.gradient(y, x)  # Overwritten by the newly calculated gradient

<tf.Tensor: shape=(4,), dtype=float32, numpy=array([1., 1., 1., 1.], dtype=float32)>